# EfficientNetV2（CIFAR100を用いた画像認識）

---
## 目的
EfficientNetV2を用いてCIFAR100データセットの100クラス物体認識を行う．EfficientNet（V1）からの変更点であるFused-MBConvと，学習の高速化・精度改善のための工夫について理解する．

## モジュールのインポート
はじめに必要なモジュールをインポートしたのち，GPUを使用した計算が可能かどうかを確認する．

In [ ]:
from time import time

import torch
import torch.nn as nn
import torchvision
import torchvision.transforms as transforms
import torchsummary

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)

## データセットの読み込みと確認
CIFAR100データセットを読み込みます．クラス数100のカラー画像データセットです．学習用データには`RandomCrop`と`RandomHorizontalFlip`によるData Augmentationを適用します．

In [ ]:
transform_train = transforms.Compose([transforms.RandomCrop(32, padding=4),
                                       transforms.RandomHorizontalFlip(),
                                       transforms.ToTensor()])
transform_test = transforms.Compose([transforms.ToTensor()])

train_data = torchvision.datasets.CIFAR100(root="./", train=True, transform=transform_train, download=True)
test_data = torchvision.datasets.CIFAR100(root="./", train=False, transform=transform_test, download=True)

print(len(train_data), len(test_data))

## EfficientNetV2の変更点（1）：Fused-MBConv
EfficientNetV2は，2021年に提案されたEfficientNetの改良版で，学習速度と精度を両立させることを目的としています．V1で使用されていたMBConv（1×1 Expansion → Depthwise $k\times k$ → SE → 1×1 Projection）は，パラメータ数・FLOPsは小さい一方，Depthwise ConvolutionはTPU/GPUのようなアクセラレータ上でメモリアクセスがボトルネックになりやすく，実際の学習・推論速度はFLOPsの小ささほど速くならないという問題がありました．

EfficientNetV2では，ネットワーク前半のステージ（チャンネル数が小さい層）に限り，Expansion（1×1）とDepthwise（$k\times k$）を1つの通常の$k\times k$畳み込みに置き換えた**Fused-MBConv**を使用します．Fused-MBConvはFLOPs・パラメータ数はMBConvより増えますが，通常の畳み込みはアクセラレータ上での実行効率が良いため，実際の学習速度が向上します．一方，チャンネル数が大きくなるネットワーク後半のステージでは，これまで通りSE付きのMBConvを使用します．つまり，「浅い層はFused-MBConv，深い層はMBConv＋SE」という使い分けがEfficientNetV2の中心的な変更点です．

In [ ]:
class SqueezeExcitation(nn.Module):
    def __init__(self, channels, se_channels):
        super().__init__()
        self.avgpool = nn.AdaptiveAvgPool2d(1)
        self.fc1 = nn.Conv2d(channels, se_channels, kernel_size=1)
        self.act1 = nn.SiLU(inplace=True)
        self.fc2 = nn.Conv2d(se_channels, channels, kernel_size=1)
        self.act2 = nn.Sigmoid()

    def forward(self, x):
        s = self.avgpool(x)
        s = self.act1(self.fc1(s))
        s = self.act2(self.fc2(s))
        return x * s


class MBConv(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size, stride, expand_ratio, se_ratio=0.25):
        super().__init__()
        self.use_residual = (stride == 1 and in_channels == out_channels)
        hidden_dim = in_channels * expand_ratio
        se_channels = max(1, int(in_channels * se_ratio))

        layers = []
        if expand_ratio != 1:
            layers += [
                nn.Conv2d(in_channels, hidden_dim, kernel_size=1, bias=False),
                nn.BatchNorm2d(hidden_dim),
                nn.SiLU(inplace=True),
            ]
        layers += [
            nn.Conv2d(hidden_dim, hidden_dim, kernel_size=kernel_size, stride=stride,
                       padding=kernel_size // 2, groups=hidden_dim, bias=False),
            nn.BatchNorm2d(hidden_dim),
            nn.SiLU(inplace=True),
        ]
        self.expand_dwconv = nn.Sequential(*layers)

        self.se = SqueezeExcitation(hidden_dim, se_channels)

        self.project = nn.Sequential(
            nn.Conv2d(hidden_dim, out_channels, kernel_size=1, bias=False),
            nn.BatchNorm2d(out_channels),
        )

    def forward(self, x):
        out = self.expand_dwconv(x)
        out = self.se(out)
        out = self.project(out)
        if self.use_residual:
            out = out + x
        return out


class FusedMBConv(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size, stride, expand_ratio):
        super().__init__()
        self.use_residual = (stride == 1 and in_channels == out_channels)
        hidden_dim = in_channels * expand_ratio

        if expand_ratio != 1:
            # Expansionと Depthwise を1つの通常の畳み込みに融合する
            self.conv = nn.Sequential(
                nn.Conv2d(in_channels, hidden_dim, kernel_size=kernel_size, stride=stride,
                           padding=kernel_size // 2, bias=False),
                nn.BatchNorm2d(hidden_dim),
                nn.SiLU(inplace=True),
            )
            self.project = nn.Sequential(
                nn.Conv2d(hidden_dim, out_channels, kernel_size=1, bias=False),
                nn.BatchNorm2d(out_channels),
            )
        else:
            # 拡張しない場合は通常の畳み込み1つのみで完結させる（Projectionは行わない）
            self.conv = nn.Sequential(
                nn.Conv2d(in_channels, out_channels, kernel_size=kernel_size, stride=stride,
                           padding=kernel_size // 2, bias=False),
                nn.BatchNorm2d(out_channels),
                nn.SiLU(inplace=True),
            )
            self.project = nn.Identity()

    def forward(self, x):
        out = self.conv(x)
        out = self.project(out)
        if self.use_residual:
            out = out + x
        return out

## ステージ構成とネットワークの作成
EfficientNetV2-Sでは，ネットワーク前半にFused-MBConv，後半にMBConv（SE付き）を配置します．本ノートブックでは，公式のEfficientNetV2-Sのステージ構成をベースに，32×32入力に合わせてストライド2のステージ数を削減し，各ステージの繰り返し数もCIFAR100での学習規模に合わせて縮小した，次の構成を実装します．

| ステージ | ブロック | 拡張率$t$ | カーネル | 出力ch | 繰り返し数 | ストライド | SE |
|---|---|---|---|---|---|---|---|
| Stem | Conv3×3 | - | 3×3 | 24 | 1 | 1 | - |
| 1 | Fused-MBConv | 1 | 3×3 | 24 | 1 | 1 | なし |
| 2 | Fused-MBConv | 4 | 3×3 | 48 | 2 | 2 | なし |
| 3 | Fused-MBConv | 4 | 3×3 | 64 | 2 | 1 | なし |
| 4 | MBConv | 4 | 3×3 | 128 | 3 | 2 | あり |
| 5 | MBConv | 6 | 3×3 | 160 | 3 | 1 | あり |
| 6 | MBConv | 6 | 3×3 | 256 | 4 | 1 | あり |
| Head | Conv1×1 | - | 1×1 | 1280 | 1 | 1 | - |

前半のFused-MBConvステージ（1〜3）ではSEを使用せず，後半のMBConvステージ（4〜6）でのみSEを使用する点がV1（全ステージでMBConv＋SE）との違いです．

In [ ]:
class EfficientNetV2S(nn.Module):
    # (block_type, 拡張率t, 出力ch, 繰り返し数, ストライド, カーネルサイズ)
    # 32x32入力に合わせてストライド2の回数を削減し，繰り返し数もCIFAR100向けに縮小している
    stage_cfg = [
        ('fused', 1, 24, 1, 1, 3),
        ('fused', 4, 48, 2, 2, 3),
        ('fused', 4, 64, 2, 1, 3),
        ('mbconv', 4, 128, 3, 2, 3),
        ('mbconv', 6, 160, 3, 1, 3),
        ('mbconv', 6, 256, 4, 1, 3),
    ]

    def __init__(self, n_class=100):
        super().__init__()
        self.stem = nn.Sequential(
            nn.Conv2d(3, 24, kernel_size=3, stride=1, padding=1, bias=False),
            nn.BatchNorm2d(24),
            nn.SiLU(inplace=True),
        )

        blocks = []
        in_channels = 24
        for block_type, expand_ratio, out_channels, repeats, stride, kernel_size in self.stage_cfg:
            for i in range(repeats):
                s = stride if i == 0 else 1
                if block_type == 'fused':
                    blocks.append(FusedMBConv(in_channels, out_channels, kernel_size, s, expand_ratio))
                else:
                    blocks.append(MBConv(in_channels, out_channels, kernel_size, s, expand_ratio))
                in_channels = out_channels
        self.blocks = nn.Sequential(*blocks)

        self.head = nn.Sequential(
            nn.Conv2d(in_channels, 1280, kernel_size=1, bias=False),
            nn.BatchNorm2d(1280),
            nn.SiLU(inplace=True),
        )
        self.avgpool = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Linear(1280, n_class)

    def forward(self, x):
        x = self.stem(x)
        x = self.blocks(x)
        x = self.head(x)
        x = self.avgpool(x)
        x = x.view(x.size(0), -1)
        x = self.fc(x)
        return x

## EfficientNetV2の変更点（2）：Progressive Learning
EfficientNetV2の論文では，ネットワーク構造の変更に加えて，**Progressive Learning**という学習手法も提案されています．これは，学習の初期は小さい入力画像サイズ・弱い正則化（Data Augmentationの強度やDropout率が低い設定）で学習を行い，学習が進むにつれて段階的に入力画像サイズと正則化の強さを大きくしていくというものです．画像サイズが小さいうちは学習が高速に進み，学習後半で画像サイズと正則化を強めることで最終的な精度を確保します．

Progressive Learningはネットワークの順伝播の構造そのものではなく学習手順（スケジューリング）に関する工夫であるため，本ノートブックのモデル定義には組み込んでいません．興味があれば課題として取り組んでみてください．

## ネットワークの作成
定義したネットワークを作成し，`.to(device)`によりモデルのパラメータを`device`上に配置します．最適化手法にはモーメンタムSGDを用い，学習率0.01，モーメンタム0.9，weight decayを5e-4とします．最後に`torchsummary.summary()`でネットワークの詳細情報を表示します．

In [ ]:
model = EfficientNetV2S(n_class=100)
model = model.to(device)

optimizer = torch.optim.SGD(model.parameters(), lr=0.01, momentum=0.9, weight_decay=5e-4)

torchsummary.summary(model, (3, 32, 32), device=device.type)

## 学習
読み込んだCIFAR100データセットと作成したネットワークを用いて学習を行います．ミニバッチサイズを128，学習エポック数を20とします．

In [ ]:
batch_size = 128
epoch_num = 20

train_loader = torch.utils.data.DataLoader(train_data, batch_size=batch_size, shuffle=True, num_workers=2)

criterion = nn.CrossEntropyLoss()
criterion = criterion.to(device)

model.train()

train_start = time()
for epoch in range(1, epoch_num + 1):
    sum_loss = 0.0
    count = 0

    for image, label in train_loader:
        image = image.to(device)
        label = label.to(device)

        y = model(image)

        loss = criterion(y, label)
        model.zero_grad()
        loss.backward()
        optimizer.step()

        sum_loss += loss.item()
        pred = torch.argmax(y, dim=1)
        count += torch.sum(pred == label)

    print("epoch: {}, mean loss: {}, mean accuracy: {}, elapsed time: {}".format(
        epoch, sum_loss / len(train_loader), count.item() / len(train_data), time() - train_start))

## テスト
学習したネットワークモデルを用いて評価を行います．

In [ ]:
test_loader = torch.utils.data.DataLoader(test_data, batch_size=100, shuffle=False)

model.eval()

count = 0
with torch.no_grad():
    for image, label in test_loader:
        image = image.to(device)
        label = label.to(device)

        y = model(image)

        pred = torch.argmax(y, dim=1)
        count += torch.sum(pred == label)

print("test accuracy: {}".format(count.item() / len(test_data)))

## ImageNet版EfficientNetV2-SとCIFAR版（本ノートブック）の違い

| | ImageNet版（V2-S） | CIFAR版（本ノートブック） |
|---|---|---|
| 入力解像度 | 384×384（学習時は可変） | 32×32 |
| Stemのストライド | 2 | 1 |
| ストライド2のステージ数 | 4 | 2 |
| 各ステージの繰り返し数 | 2, 4, 4, 6, 9, 15 | 1, 2, 2, 3, 3, 4 |
| Progressive Learning | あり（画像サイズ・正則化を段階的に強化） | なし（固定サイズ・固定Augmentationで学習） |
| 出力層のクラス数 | 1000 | 100（CIFAR100） |

Fused-MBConvを前半，MBConv＋SEを後半に配置するというステージ構成の考え方自体はオリジナルと同じですが，繰り返し数はCIFAR100での学習規模に合わせて縮小しています．

## 課題

1. Stage4〜6のMBConv（SE付き）を，V1と同様にすべてのステージでSE付きMBConvに置き換えたモデルと，本ノートブックのFused-MBConv/MBConvを併用したモデルとで，パラメータ数・1エポックあたりの学習時間・認識精度を比較しましょう．

2. 反対に，全ステージをFused-MBConvに置き換えたモデルを作成し，SEを使わないことによる精度への影響を確認しましょう．

3. Progressive Learningのアイデアを参考に，学習の前半は`RandomCrop`のpadding幅を小さくし，後半に大きくするなど，学習中にAugmentationの強度を変化させる実験をしてみましょう．